# JPA Live Tracker - SMPD v2 Run30

Continuously re-checks a selected pump-gain/DC-bias range and updates the best JPA point. Stop the continuous loop with kernel interrupt.

## Imports

In [ ]:
import os
import time
import importlib
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from qick import QickConfig

from slab_qick_calib.exp_handling.instrumentmanager import InstrumentManager
import slab_qick_calib.experiments as meas
from slab_qick_calib.helpers import config, rfboard, handy

importlib.reload(meas)
handy.config_figs()
%config InlineBackend.figure_format = 'png'


## Connect

In [ ]:
cfg_file = "2026_06_08_smpd_v2_run30.yml"
expt_path = r"C:\_Data\SMPD\2026_06_01_Yuvi_v2_run30"

repo_root = Path.cwd().resolve()
while not (repo_root / "configs").exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent

cfg_path = str(repo_root / "configs" / cfg_file)
images_dir = os.path.join(expt_path, "images")
data_dir = os.path.join(expt_path, "data")
summary_dir = os.path.join(images_dir, "summary")
for folder in (expt_path, images_dir, data_dir, summary_dir):
    os.makedirs(folder, exist_ok=True)

auto_cfg = config.load(cfg_path)
im = InstrumentManager(ns_address=auto_cfg["aliases"]["ip"], port=9090)
soc = im[auto_cfg["aliases"]["soc"]]
soccfg = QickConfig(soc.get_cfg())
cfg_dict = {"soc": soccfg, "expt_path": expt_path, "cfg_file": cfg_path, "im": im}

print(f"Config: {cfg_path}")
print(f"Data:   {expt_path}")


## Tracker Settings

In [ ]:
qi = 1
auto_cfg = rfboard.activate_qubit_rf(qi, soc, auto_cfg, cfg_file=cfg_path)
auto_cfg = config.load(cfg_path)

WASTE_RESONANCE_MHZ = 7780.828
PUMP_DETUNING_MHZ = 2.0
PUMP_FREQ_MHZ = WASTE_RESONANCE_MHZ + PUMP_DETUNING_MHZ
PUMP_CH = 13
PUMP_NQZ = 2
PUMP_IF_MHZ = 400.0
PUMP_MIXER_FREQ_MHZ = PUMP_FREQ_MHZ - PUMP_IF_MHZ
PUMP_ATTEN1_DB = 15
PUMP_ATTEN2_DB = 0
PUMP_GAIN_MAX = 1.0
PUMP_FILTER_BW_MHZ = 1000.0
PUMP_FILTER_TYPE = "bandpass"

JPA_DC_CH = auto_cfg.hw.soc.dacs.flux.dc_ch[qi]
JPA_OFF_BIAS_V = 0.0
DC_SETTLE_S = 0.25

pump_range = (0.02, 0.12)
dc_range = (112e-3, 122e-3)
adaptive_stage_grids = [(5, 5), (3, 3), (3, 3)]
adaptive_shrink = 0.5
adaptive_cycles_per_pass = 1

tracker_update_yaml = True
tracker_sleep_s = 0.0

print(f"Waste: {WASTE_RESONANCE_MHZ:.3f} MHz")
print(f"Pump:  {PUMP_FREQ_MHZ:.3f} MHz, mixer {PUMP_MIXER_FREQ_MHZ:.3f} MHz")
print(f"Pump range: {pump_range}")
print(f"DC range: {1e3*dc_range[0]:.3f} to {1e3*dc_range[1]:.3f} mV")
print(f"YAML update: {tracker_update_yaml}")


## One Tracker Pass

In [ ]:
jpa_tracker = meas.JPAAdaptiveFineTuning(
    cfg_dict,
    qi=qi,
    params={
        "waste_freq_mhz": WASTE_RESONANCE_MHZ,
        "center": WASTE_RESONANCE_MHZ,
        "span": 20.0,
        "expts": 301,
        "gain": 0.01,
        "length": 10,
        "reps": 2000,
        "rounds": 1,
        "pump_ch": PUMP_CH,
        "pump_nqz": PUMP_NQZ,
        "pump_freq_mhz": PUMP_FREQ_MHZ,
        "pump_mixer_freq_mhz": PUMP_MIXER_FREQ_MHZ,
        "pump_attn1_db": PUMP_ATTEN1_DB,
        "pump_attn2_db": PUMP_ATTEN2_DB,
        "pump_gain_max": PUMP_GAIN_MAX,
        "pump_filter_bw_mhz": PUMP_FILTER_BW_MHZ,
        "pump_filter_type": PUMP_FILTER_TYPE,
        "pump_range": pump_range,
        "dc_range": dc_range,
        "pump_gains": np.linspace(*pump_range, 5),
        "dc_biases_v": np.linspace(*dc_range, 5),
        "adaptive_stage_grids": adaptive_stage_grids,
        "adaptive_shrink": adaptive_shrink,
        "adaptive_cycles": adaptive_cycles_per_pass,
        "adaptive_full_rescan_each_cycle": True,
        "adaptive_live": True,
        "show_baseline": True,
        "bias_chan": JPA_DC_CH,
        "off_bias": JPA_OFF_BIAS_V,
        "dc_settle": DC_SETTLE_S,
        "gain_window_mhz": 0.5,
        "rspec_update": tracker_update_yaml,
    },
)

print(f"Best gain: {jpa_tracker.data['best_gain_db']:.2f} dB")
print(f"Best pump gain: {jpa_tracker.data['best_pump_gain']:.5f}")
print(f"Best DC bias: {jpa_tracker.data['best_dc_bias_mv']:.3f} mV")


## Continuous Tracker

This cell runs forever unless `max_tracker_passes` is set to an integer. Interrupt the kernel to stop it.

In [ ]:
tracker_history = []
max_tracker_passes = None
pass_i = 0

try:
    while max_tracker_passes is None or pass_i < max_tracker_passes:
        print(f"JPA tracker pass {pass_i}")
        jpa_tracker = meas.JPAAdaptiveFineTuning(
            cfg_dict,
            qi=qi,
            params={
                "waste_freq_mhz": WASTE_RESONANCE_MHZ,
                "center": WASTE_RESONANCE_MHZ,
                "span": 20.0,
                "expts": 301,
                "gain": 0.01,
                "length": 10,
                "reps": 2000,
                "rounds": 1,
                "pump_ch": PUMP_CH,
                "pump_nqz": PUMP_NQZ,
                "pump_freq_mhz": PUMP_FREQ_MHZ,
                "pump_mixer_freq_mhz": PUMP_MIXER_FREQ_MHZ,
                "pump_attn1_db": PUMP_ATTEN1_DB,
                "pump_attn2_db": PUMP_ATTEN2_DB,
                "pump_gain_max": PUMP_GAIN_MAX,
                "pump_filter_bw_mhz": PUMP_FILTER_BW_MHZ,
                "pump_filter_type": PUMP_FILTER_TYPE,
                "pump_range": pump_range,
                "dc_range": dc_range,
                "pump_gains": np.linspace(*pump_range, 5),
                "dc_biases_v": np.linspace(*dc_range, 5),
                "adaptive_stage_grids": adaptive_stage_grids,
                "adaptive_shrink": adaptive_shrink,
                "adaptive_cycles": adaptive_cycles_per_pass,
                "adaptive_full_rescan_each_cycle": True,
                "adaptive_live": True,
                "show_baseline": True,
                "bias_chan": JPA_DC_CH,
                "off_bias": JPA_OFF_BIAS_V,
                "dc_settle": DC_SETTLE_S,
                "gain_window_mhz": 0.5,
                "rspec_update": tracker_update_yaml,
            },
        )
        tracker_history.append({
            "pass": pass_i,
            "best_gain_db": jpa_tracker.data["best_gain_db"],
            "best_pump_gain": jpa_tracker.data["best_pump_gain"],
            "best_dc_bias_v": jpa_tracker.data["best_dc_bias_v"],
            "data": jpa_tracker.data,
        })
        print(
            f"Pass {pass_i}: gain {jpa_tracker.data['best_gain_db']:.2f} dB, "
            f"pump {jpa_tracker.data['best_pump_gain']:.5f}, "
            f"DC {jpa_tracker.data['best_dc_bias_mv']:.3f} mV"
        )
        pass_i += 1
        if tracker_sleep_s > 0:
            time.sleep(tracker_sleep_s)
except KeyboardInterrupt:
    print("Stopped JPA tracker.")
